# Plant Disease Classification - PlantDoc + PlantVillage (v2)

Same **eight diseases** as before, but now trained on **two datasets combined**:

* **PlantDoc** - photos taken in the wild (messy backgrounds). Only ~750 training images.
* **PlantVillage** - ~3,000 clean, single-leaf images (via `tensorflow_datasets`, no Kaggle key needed).

Training on both gives the models far more examples. Because PlantVillage is a much *easier* domain, results are reported on **two separate test sets**:

| Test set | What it tells you |
|---|---|
| **PlantDoc test (original 76 images)** | Realistic, in-the-wild performance. **This is the headline number.** |
| **PlantVillage test** | Lab-style images. Expect ~95%+, but near-duplicate leaves exist across PlantVillage splits, so treat it as an optimistic upper bound. |

Model selection (early stopping, best-model choice) uses a **PlantDoc validation split only**, so it optimizes for the real-world target.

> Use **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# 1. Imports, seed, GPU and mixed precision
import os, json, shutil, time, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from PIL import Image

SEED = 42
tf.keras.utils.set_random_seed(SEED)

USE_MIXED_PRECISION = True   # faster on T4/A100; set False if you ever see NaN losses
gpus = tf.config.list_physical_devices("GPU")
if gpus and USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow:", tf.__version__)
print("GPU:", gpus)
print("Policy:", tf.keras.mixed_precision.global_policy().name)

TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Policy: mixed_float16


In [ ]:
# 2. Download PlantDoc (shallow clone) and check the PlantVillage loader
ROOT = "/content/PlantDoc-Dataset"
if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/pratikkayal/PlantDoc-Dataset.git", ROOT], check=True)
print("PlantDoc found:", os.path.exists(ROOT), "| folders:", os.listdir(ROOT))
import tensorflow_datasets as tfds
print("tensorflow_datasets:", tfds.__version__)

PlantDoc found: True | folders: ['README.md', 'LICENSE.txt', 'test', '.git', 'PlantDoc_Examples.png', 'train']
tensorflow_datasets: 4.9.10


In [ ]:
# 3. Automatically select eight real disease classes from the dataset
# This avoids guessing folder names. Only folders present in BOTH train and test are eligible.
train_root = os.path.join(ROOT, "train")
test_root = os.path.join(ROOT, "test")
if not os.path.isdir(train_root) or not os.path.isdir(test_root):
    raise FileNotFoundError(
        f"Expected train/ and test/ folders inside {ROOT}. "
        "Check that the PlantDoc dataset downloaded correctly."
    )

train_folders = {name for name in os.listdir(train_root)
                 if os.path.isdir(os.path.join(train_root, name))}
test_folders = {name for name in os.listdir(test_root)
                if os.path.isdir(os.path.join(test_root, name))}
available_classes = sorted(train_folders & test_folders)

# These words indicate disease-related labels in typical PlantDoc class names.
disease_keywords = (
    "blight", "spot", "mildew", "rust", "virus", "mosaic", "scab",
    "rot", "canker", "curl", "scorch", "bacterial", "anthracnose",
    "wilt", "mold", "mould", "mite", "leaf miner", "septoria",
    "powdery", "downy"
)

def looks_like_disease_class(class_name):
    name = class_name.lower()
    return any(word in name for word in disease_keywords)

# Keep the five disease classes already used in the project when present.
preferred_disease_classes = [
    "Corn leaf blight",
    "Tomato Septoria leaf spot",
    "Squash Powdery mildew leaf",
    "Potato leaf early blight",
    "Corn rust leaf",
]
selected_classes = [name for name in preferred_disease_classes
                    if name in available_classes]

# Fill the remaining places using exact folder names found in this dataset.
for name in available_classes:
    if name not in selected_classes and looks_like_disease_class(name):
        selected_classes.append(name)
    if len(selected_classes) == 8:
        break

if len(selected_classes) < 8:
    disease_candidates = [name for name in available_classes if looks_like_disease_class(name)]
    raise ValueError(
        f"Found only {len(selected_classes)} suitable disease classes, but this notebook needs 8. "
        f"Available disease-like folders: {disease_candidates}. "
        "Edit disease_keywords or select eight valid folder names manually."
    )

print("Selected disease classes:", len(selected_classes))
for i, cls in enumerate(selected_classes, start=1):
    train_count = len([f for f in os.listdir(os.path.join(train_root, cls))
                       if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))])
    test_count = len([f for f in os.listdir(os.path.join(test_root, cls))
                      if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))])
    print(f"{i}. {cls} | train images: {train_count} | test images: {test_count}")

Selected disease classes: 8
1. Corn leaf blight | train images: 180 | test images: 12
2. Tomato Septoria leaf spot | train images: 140 | test images: 11
3. Squash Powdery mildew leaf | train images: 124 | test images: 6
4. Potato leaf early blight | train images: 109 | test images: 8
5. Corn rust leaf | train images: 106 | test images: 10
6. Apple Scab Leaf | train images: 83 | test images: 10
7. Apple rust leaf | train images: 79 | test images: 10
8. Bell_pepper leaf spot | train images: 62 | test images: 9


In [ ]:

%pip install -U importlib_resources tensorflow-datasets


In [ ]:

# 4. Extract the same 8 diseases from PlantVillage

PV_MATCH = {
    "Corn leaf blight": ("corn", "northern"),
    "Tomato Septoria leaf spot": ("tomato", "septoria"),
    "Squash Powdery mildew leaf": ("squash", "powdery"),
    "Potato leaf early blight": ("potato", "early blight"),
    "Corn rust leaf": ("corn", "common rust"),
    "Apple Scab Leaf": ("apple", "scab"),
    "Apple rust leaf": ("apple", "cedar"),
    "Bell_pepper leaf spot": ("pepper", "bacterial spot"),
}

PV_CAP_PER_CLASS = 400
PV_RAW = "/content/PlantVillage_raw"

# Validate selected classes
missing = [c for c in selected_classes if c not in PV_MATCH]
if missing:
    raise ValueError(f"Missing PlantVillage mappings: {missing}")

# Load PlantVillage dataset
pv_ds, pv_info = tfds.load(
    "plant_village",
    split="train",
    with_info=True,
    shuffle_files=False,
    decoders={"image": tfds.decode.SkipDecoding()},
)

pv_names = pv_info.features["label"].names

# Normalize labels for matching
def normalize_label(label):
    return label.lower().replace("_", " ").replace("-", " ")

label_to_cls = {}

for cls in selected_classes:
    keywords = PV_MATCH[cls]

    hits = [
        i for i, name in enumerate(pv_names)
        if all(k in normalize_label(name) for k in keywords)
    ]

    if len(hits) != 1:
        raise ValueError(
            f"{cls}: expected exactly 1 match, found "
            f"{[pv_names[i] for i in hits]}\n"
            f"Available labels: {pv_names}"
        )

    label_to_cls[hits[0]] = cls
    print(f"{cls:<30} <- {pv_names[hits[0]]}")

# Filter only selected labels
keys = tf.constant(list(label_to_cls.keys()), dtype=tf.int64)

pv_ds = pv_ds.filter(
    lambda ex: tf.reduce_any(
        tf.equal(tf.cast(ex["label"], tf.int64), keys)
    )
)

pv_ds = pv_ds.shuffle(
    buffer_size=8000,
    seed=SEED,
    reshuffle_each_iteration=False,
)

# Create output folders
if os.path.exists(PV_RAW):
    shutil.rmtree(PV_RAW)

for cls in selected_classes:
    os.makedirs(os.path.join(PV_RAW, cls), exist_ok=True)

# Extract images with per-class limits
kept = {cls: 0 for cls in selected_classes}

for ex in tfds.as_numpy(pv_ds):
    label = int(ex["label"])
    cls = label_to_cls[label]

    if kept[cls] >= PV_CAP_PER_CLASS:
        continue

    image_path = os.path.join(
        PV_RAW, cls, f"pv_{kept[cls]:04d}.jpg"
    )

    with open(image_path, "wb") as f:
        f.write(ex["image"])

    kept[cls] += 1

    if all(
        count >= PV_CAP_PER_CLASS
        for count in kept.values()
    ):
        break

print("PlantVillage images kept per class:", kept)
print("Total images kept:", sum(kept.values()))


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.HBRPOI_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
Corn leaf blight               <- Corn___Northern_Leaf_Blight
Tomato Septoria leaf spot      <- Tomato___Septoria_leaf_spot
Squash Powdery mildew leaf     <- Squash___Powdery_mildew
Potato leaf early blight       <- Potato___Early_blight
Corn rust leaf                 <- Corn___Common_rust
Apple Scab Leaf                <- Apple___Apple_scab
Apple rust leaf                <- Apple___Cedar_apple_rust
Bell_pepper leaf spot          <- Pepper,_bell___Bacterial_spot
PlantVillage images kept per class: {'Corn leaf blight': 400, 'Tomato Septoria leaf spot': 400, 'Squash Powdery mildew leaf': 400, 'Potato leaf early blight': 400, 'Corn rust leaf': 400, 'Apple Scab Leaf': 400, 'Apple rust leaf': 275, 'Bell_pepper leaf spot': 400}
Total images kept: 3075


In [ ]:
# 5. Build splits. Validation + the honest test set both come from PlantDoc.
SPLIT_DIR = "/content/PlantDisease_Split_v2"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
if os.path.exists(SPLIT_DIR): shutil.rmtree(SPLIT_DIR)

def copy_files(files, src_dir, split, cls):
    dest = os.path.join(SPLIT_DIR, split, cls); os.makedirs(dest, exist_ok=True)
    for fn in files: shutil.copy2(os.path.join(src_dir, fn), os.path.join(dest, fn))
    return len(files)

counts = {"pd_train": 0, "pd_val": 0, "pd_test": 0, "pv_train": 0, "pv_test": 0}
for cls in selected_classes:
    # PlantDoc: original test stays untouched; validation is carved out of original train
    tr_dir, te_dir = os.path.join(ROOT, "train", cls), os.path.join(ROOT, "test", cls)
    tr = sorted(f for f in os.listdir(tr_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    te = sorted(f for f in os.listdir(te_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    tr, va = train_test_split(tr, test_size=0.15, random_state=SEED)
    counts["pd_train"] += copy_files(tr, tr_dir, "pd_train", cls)
    counts["pd_val"]   += copy_files(va, tr_dir, "pd_val", cls)
    counts["pd_test"]  += copy_files(te, te_dir, "pd_test", cls)
    # PlantVillage: 85% train / 15% (optimistic) test
    pv_dir = os.path.join(PV_RAW, cls)
    pv = sorted(os.listdir(pv_dir))
    pv_tr, pv_te = train_test_split(pv, test_size=0.15, random_state=SEED)
    counts["pv_train"] += copy_files(pv_tr, pv_dir, "pv_train", cls)
    counts["pv_test"]  += copy_files(pv_te, pv_dir, "pv_test", cls)
print("Split complete:", counts)

Split complete: {'pd_train': 748, 'pd_val': 135, 'pd_test': 76, 'pv_train': 2613, 'pv_test': 462}


In [ ]:
# 6. Fast input pipeline: decode once -> cache as uint8 -> mix both domains -> per-sample shuffle
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
PD_REPEAT = 3     # PlantDoc is the real target but small: show each PlantDoc image 3x per epoch (augmentation makes each pass different)

class_names = sorted(selected_classes)
NUM_CLASSES = len(class_names)

def load_split(name):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_DIR, name), image_size=IMG_SIZE, batch_size=None,
        label_mode="categorical", class_names=class_names, shuffle=False)

pv_train_raw, pd_train_raw = load_split("pv_train"), load_split("pd_train")
pd_val_raw, pd_test_raw, pv_test_raw = load_split("pd_val"), load_split("pd_test"), load_split("pv_test")
print("Classes:", class_names)

def to_uint8(x, y): return tf.cast(tf.round(x), tf.uint8), y
def to_float(x, y): return tf.cast(x, tf.float32), y

def cached(ds): return ds.map(to_uint8, num_parallel_calls=AUTOTUNE).cache()
def finish(ds):  return ds.batch(BATCH_SIZE).map(to_float, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

n_train = len(pv_train_raw.file_paths) + PD_REPEAT * len(pd_train_raw.file_paths)
train_ds = finish(cached(pv_train_raw).concatenate(cached(pd_train_raw).repeat(PD_REPEAT))
                  .shuffle(n_train, seed=SEED, reshuffle_each_iteration=True))
val_ds      = finish(cached(pd_val_raw))      # PlantDoc-only validation = real-world target
test_pd_ds  = finish(cached(pd_test_raw))     # headline test set
test_pv_ds  = finish(cached(pv_test_raw))     # optimistic lab-style test set
EVAL_SETS = {"PlantDoc test": test_pd_ds, "PlantVillage test": test_pv_ds}
PRIMARY = "PlantDoc test"

# Class weights over the effective training distribution
def labels_of(raw):
    return np.array([class_names.index(os.path.basename(os.path.dirname(p))) for p in raw.file_paths])
all_labels = np.concatenate([labels_of(pv_train_raw)] + [labels_of(pd_train_raw)] * PD_REPEAT)
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=all_labels)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print("Training samples per epoch:", n_train)
print("Class weights:", {class_names[i]: round(w, 2) for i, w in class_weight.items()})

Found 2613 files belonging to 8 classes.
Found 748 files belonging to 8 classes.
Found 135 files belonging to 8 classes.
Found 76 files belonging to 8 classes.
Found 462 files belonging to 8 classes.
Classes: ['Apple Scab Leaf', 'Apple rust leaf', 'Bell_pepper leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Potato leaf early blight', 'Squash Powdery mildew leaf', 'Tomato Septoria leaf spot']
Training samples per epoch: 4857
Class weights: {'Apple Scab Leaf': 1.1, 'Apple rust leaf': 1.4, 'Bell_pepper leaf spot': 1.22, 'Corn leaf blight': 0.76, 'Corn rust leaf': 1.0, 'Potato leaf early blight': 0.99, 'Squash Powdery mildew leaf': 0.93, 'Tomato Septoria leaf spot': 0.87}


In [ ]:
# 7. Shared helpers: augmentation, callbacks, training, evaluation
def make_augmentation():
    # Runs on the GPU inside the model, and is automatically disabled at inference time.
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15, fill_mode="reflect"),
        layers.RandomZoom(0.2, fill_mode="reflect"),
        layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
    ], name="augmentation")

def make_loss():
    return tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

def make_callbacks(ckpt_path, patience):
    return [
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=max(2, patience // 2),
                          min_lr=1e-7, verbose=1),
        ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
    ]

def merge_history(*histories):
    merged = {}
    for h in histories:
        for k, v in h.items():
            merged.setdefault(k, []).extend(v)
    return merged

# ---- Evaluation with optional test-time augmentation (average over flipped copies) ----
TTA_TRANSFORMS = [
    lambda x: x,
    lambda x: tf.image.flip_left_right(x),
    lambda x: tf.image.flip_up_down(x),
    lambda x: tf.image.flip_up_down(tf.image.flip_left_right(x)),
]

def predict_probs(model, ds, tta=True):
    transforms = TTA_TRANSFORMS if tta else TTA_TRANSFORMS[:1]
    probs = [model.predict(ds.map(lambda x, y, t=t: (t(x), y)), verbose=0) for t in transforms]
    return np.mean(probs, axis=0)

results = {}   # name -> {"PlantDoc test": {...}, "PlantVillage test": {...}, "val_acc": float}

def evaluate_model(name, model, labels, tta=True):
    results[name] = {}
    for key, ds in EVAL_SETS.items():
        y_true = np.argmax(np.concatenate([y.numpy() for _, y in ds]), axis=1)
        acc_plain = accuracy_score(y_true, np.argmax(predict_probs(model, ds, tta=False), axis=1))
        y_pred = np.argmax(predict_probs(model, ds, tta=tta), axis=1)
        scores = {
            "Accuracy":  accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1-Score":  f1_score(y_true, y_pred, average="weighted", zero_division=0),
        }
        print(f"[{name}] {key}: accuracy without TTA {acc_plain:.4f} | with TTA {scores['Accuracy']:.4f}")
        if key == PRIMARY:
            print(pd.Series(scores).round(4))
            print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=labels, zero_division=0))
        results[name][key] = {"true": y_true, "pred": y_pred, "scores": scores,
                              "cm": confusion_matrix(y_true, y_pred)}
    yv = np.argmax(np.concatenate([y.numpy() for _, y in val_ds]), axis=1)
    results[name]["val_acc"] = accuracy_score(yv, np.argmax(predict_probs(model, val_ds, tta=False), axis=1))
    return results[name]

# ---- Transfer-learning builder + 2-stage trainer ----
def build_transfer(base_fn, name, rescale_to_pm1=False):
    inputs = layers.Input(shape=(224, 224, 3))
    x = make_augmentation()(inputs)
    if rescale_to_pm1:                                   # MobileNetV2 expects [-1, 1]
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)    # (EfficientNetV2 rescales internally)
    base = base_fn(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    base.trainable = False
    x = base(x, training=False)                          # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return models.Model(inputs, outputs, name=name), base

def train_two_stage(model, base, name, unfreeze_last, head_epochs=12, ft_epochs=25, head_lr=1e-3, ft_lr=2e-5):
    t0 = time.time()
    # Stage 1: train only the new classification head
    model.compile(optimizer=tf.keras.optimizers.Adam(head_lr), loss=make_loss(), metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, class_weight=class_weight,
                   callbacks=[EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                              ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, verbose=1)])
    # Stage 2: unfreeze the top of the backbone (BatchNorm stays frozen) and fine-tune at a tiny LR
    base.trainable = True
    for layer in base.layers[:-unfreeze_last]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(ft_lr), loss=make_loss(), metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs, class_weight=class_weight,
                   callbacks=make_callbacks(f"/content/best_{name}.keras", patience=8))
    print(f"[{name}] training time: {(time.time() - t0) / 60:.1f} min")
    return merge_history(h1.history, h2.history), len(h1.history["loss"])

histories, finetune_start = {}, {}

## Model 1: Custom CNN

In [ ]:
# 8. Build and train the CNN baseline (BatchNorm + GlobalAveragePooling instead of a giant Dense layer)
def conv_block(x, filters, drop=0.1):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(drop)(x)

inputs = layers.Input(shape=(224, 224, 3))
x = make_augmentation()(inputs)
x = layers.Rescaling(1 / 255.0)(x)
for f in (32, 64, 128, 256):
    x = conv_block(x, f)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
cnn_model = models.Model(inputs, outputs, name="custom_cnn")

cnn_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=make_loss(), metrics=["accuracy"])
cnn_model.summary()

t0 = time.time()
cnn_hist = cnn_model.fit(train_ds, validation_data=val_ds, epochs=40, class_weight=class_weight,
                         callbacks=make_callbacks("/content/best_cnn_model.keras", patience=8))
print(f"CNN training time: {(time.time() - t0) / 60:.1f} min")
histories["CNN"] = cnn_hist.history
finetune_start["CNN"] = None

In [ ]:
# 9. Evaluate CNN on both test sets
evaluate_model("CNN", cnn_model, class_names);

## Model 2: MobileNetV2 (transfer learning + fine-tuning)

In [ ]:
# 10. Build and train MobileNetV2 (stage 1: head, stage 2: fine-tune top layers)
mobilenet_model, mobilenet_base = build_transfer(
    tf.keras.applications.MobileNetV2, "mobilenetv2", rescale_to_pm1=True)
mobilenet_model.summary()
histories["MobileNetV2"], finetune_start["MobileNetV2"] = train_two_stage(
    mobilenet_model, mobilenet_base, "mobilenetv2", unfreeze_last=60)

In [ ]:
# 11. Evaluate MobileNetV2 on both test sets
evaluate_model("MobileNetV2", mobilenet_model, class_names);

## Model 3: EfficientNetV2-B0 (transfer learning + fine-tuning)

In [ ]:
# 12. Build and train EfficientNetV2-B0 (it rescales 0-255 inputs internally)
effnet_model, effnet_base = build_transfer(tf.keras.applications.EfficientNetV2B0, "efficientnetv2b0")
effnet_model.summary()
histories["EfficientNetV2B0"], finetune_start["EfficientNetV2B0"] = train_two_stage(
    effnet_model, effnet_base, "efficientnetv2b0", unfreeze_last=100)

In [ ]:
# 13. Evaluate EfficientNetV2-B0 on both test sets
evaluate_model("EfficientNetV2B0", effnet_model, class_names);

In [ ]:
# 14. Compare models: PlantDoc (headline) and PlantVillage (optimistic) results
model_names = list(results)
comparison = pd.DataFrame({"Metric": list(results[model_names[0]][PRIMARY]["scores"])})
for n in model_names:
    comparison[f"{n} (%)"] = [v * 100 for v in results[n][PRIMARY]["scores"].values()]
print(f"=== {PRIMARY.upper()} (headline, in-the-wild images) ===")
display(comparison.round(2))
comparison.to_csv("/content/final_model_comparison.csv", index=False)

pv_table = pd.DataFrame({"Model": model_names,
                         "PlantDoc test acc (%)": [results[n][PRIMARY]["scores"]["Accuracy"] * 100 for n in model_names],
                         "PlantVillage test acc (%)": [results[n]["PlantVillage test"]["scores"]["Accuracy"] * 100 for n in model_names],
                         "PlantDoc val acc (%)": [results[n]["val_acc"] * 100 for n in model_names]})
print("\n=== Accuracy on every split ===")
display(pv_table.round(2))
pv_table.to_csv("/content/accuracy_all_splits.csv", index=False)

x = np.arange(len(comparison)); w = 0.8 / len(model_names)
plt.figure(figsize=(10, 5))
for i, n in enumerate(model_names):
    plt.bar(x + (i - (len(model_names) - 1) / 2) * w, comparison[f"{n} (%)"], w, label=n)
plt.xticks(x, comparison["Metric"])
plt.ylabel("Score (%)"); plt.ylim(0, 100); plt.title("Model comparison - PlantDoc test set (with TTA)")
plt.legend(); plt.tight_layout(); plt.savefig("/content/model_comparison.png", dpi=200); plt.show()

In [ ]:
# 15. Display and save confusion matrices
fig, axes = plt.subplots(1, len(model_names), figsize=(8 * len(model_names), 7))
for ax, n in zip(np.atleast_1d(axes), model_names):
    matrix = results[n][PRIMARY]["cm"]
    ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"{n} - {PRIMARY} confusion matrix")
    ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
    ax.set_xticks(range(NUM_CLASSES), class_names, rotation=90)
    ax.set_yticks(range(NUM_CLASSES), class_names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, matrix[i, j], ha="center", va="center", fontsize=8,
                    color="white" if matrix[i, j] > matrix.max() / 2 else "black")
fig.tight_layout(); fig.savefig("/content/confusion_matrices.png", dpi=200); plt.show()

for n in model_names:
    pd.DataFrame(results[n][PRIMARY]["cm"], index=class_names, columns=class_names).to_csv(f"/content/{n.lower()}_confusion_matrix.csv")
    pd.DataFrame(classification_report(results[n][PRIMARY]["true"], results[n][PRIMARY]["pred"], target_names=class_names,
                                       output_dict=True, zero_division=0)).T.to_csv(f"/content/{n.lower()}_classification_report.csv")

In [ ]:
# 16. Plot training / validation accuracy and loss (dashed line = start of fine-tuning)
fig, axes = plt.subplots(2, len(histories), figsize=(6 * len(histories), 8), squeeze=False)
for col, (n, h) in enumerate(histories.items()):
    for row, (key, title) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
        ax = axes[row, col]
        ax.plot(h[key], label="train"); ax.plot(h["val_" + key], label="validation")
        if finetune_start[n]:
            ax.axvline(finetune_start[n] - 0.5, color="gray", linestyle="--", label="fine-tune start")
        ax.set_title(f"{n} - {title}"); ax.set_xlabel("Epoch"); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); plt.savefig("/content/training_history.png", dpi=200); plt.show()

In [ ]:
from google.colab import drive
import json, os, shutil, glob
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/PlantDisease_Demo"
os.makedirs(SAVE_DIR, exist_ok=True)

trained = {"CNN": cnn_model, "MobileNetV2": mobilenet_model, "EfficientNetV2B0": effnet_model}

# 1) Models seedha Drive mein
for n, m in trained.items():
    m.save(f"{SAVE_DIR}/final_{n.lower()}_model.keras")

# 2) Class names
with open(f"{SAVE_DIR}/plant_disease_class_names.json", "w") as f:
    json.dump(list(class_names), f)

# 3) Config (demo notebook ke liye)
best_name = max(results, key=lambda n: results[n]["val_acc"])
cfg = {
    "best_name": best_name,
    "img_size": IMG_SIZE if isinstance(IMG_SIZE, int) else list(IMG_SIZE),
    "class_names": list(class_names),
    "val_acc": {n: float(r["val_acc"]) for n, r in results.items()},
}
json.dump(cfg, open(f"{SAVE_DIR}/config.json", "w"), indent=2)

# 4) Jo CSV/PNG/report files /content mein bani hain, wo bhi copy karo
for pattern in ["/content/*.csv", "/content/*.png", "/content/*.txt", "/content/*.json"]:
    for f in glob.glob(pattern):
        shutil.copy(f, SAVE_DIR)

print("Drive mein saved files:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(" -", f)

## Single-image prediction demo
Upload a leaf image. The model with the best **PlantDoc validation accuracy** shows its top three predictions among the **eight selected disease classes** (with test-time augmentation). Confidence is the model's score, not a guarantee of a correct diagnosis; other plants/diseases may be misclassified.

In [ ]:
import os

model_path = "/content/drive/MyDrive/PlantDisease_Demo"

print(os.listdir(model_path))

In [ ]:
import tensorflow as tf
import os
import json

model_path = "/content/drive/MyDrive/PlantDisease_Demo"

mobilenet_model = tf.keras.models.load_model(
    os.path.join(model_path, "final_mobilenetv2_model.keras")
)

with open(os.path.join(model_path, "plant_disease_class_names.json"), "r") as f:
    class_names = json.load(f)

print("Model loaded successfully!")
print("Classes:", class_names)

In [ ]:
# Show only the Top 1 prediction

import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

SAVE_DIR = "/content/drive/MyDrive/PlantDisease_Demo"
config_path = os.path.join(SAVE_DIR, "config.json")

if not os.path.exists(config_path):
    raise FileNotFoundError(
        f"Could not find config.json at {SAVE_DIR}. "
        "Please ensure your Google Drive is mounted and step 16 ran successfully."
    )

# Load configuration and classes
with open(config_path, "r") as f:
    cfg = json.load(f)

best_name = cfg["best_name"]
class_names = cfg["class_names"]
IMG_SIZE = cfg["img_size"]

# Load the saved model directly from Google Drive
model_path = os.path.join(
    SAVE_DIR, f"final_{best_name.lower()}_model.keras"
)
best_model = tf.keras.models.load_model(model_path)

# TTA Transforms helper
TTA_TRANSFORMS = [
    lambda x: x,
    lambda x: tf.image.flip_left_right(x),
    lambda x: tf.image.flip_up_down(x),
    lambda x: tf.image.flip_up_down(tf.image.flip_left_right(x)),
]

uploaded = files.upload()

if uploaded:
    image_path = next(iter(uploaded))
    image = Image.open(image_path).convert("RGB")

    x = np.asarray(
        image.resize((IMG_SIZE[1], IMG_SIZE[0])),
        dtype=np.float32
    )

    batch = tf.constant(np.expand_dims(x, axis=0))

    predictions = [
        best_model.predict(transform(batch), verbose=0)[0]
        for transform in TTA_TRANSFORMS
    ]

    probabilities = np.mean(predictions, axis=0)
    idx = np.argmax(probabilities)
    confidence = probabilities[idx] * 100

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    print(f"Predicted disease: {class_names[idx]}")
    print(f"Confidence: {confidence:.2f}%")

    if confidence >= 80:
        print("High model confidence.")
    else:
        print("Low confidence. Try a clearer, well-lit leaf image.")

CLOUD DEPLOYMENT CHECKS


In [ ]:

import os

for root, dirs, files in os.walk("/content/PlantDisease_Split_v2"):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]
    class_dirs = [
        d for d in dirs
        if any(os.path.isdir(os.path.join(root, d, x)) for x in [])
    ]
    if class_dirs:
        print(root, class_dirs)

print("\nModel exists:", os.path.exists("/content/best_mobilenetv2.keras"))


In [ ]:
import os

for root, dirs, files in os.walk("/content/PlantDisease_Split_v2"):
    if root.count(os.sep) <= 4:
        print(root, "->", dirs[:15])

In [ ]:

import os
import json
import zipfile
import tensorflow as tf
from google.colab import files

# 1. Load your existing trained model
model_path = "/content/best_mobilenetv2.keras"
model = tf.keras.models.load_model(model_path)

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

# 2. Extract the eight class names
dataset_path = "/content/PlantDisease_Split_v2/pv_train"

class_names = sorted([
    name for name in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, name))
])

print("Number of classes:", len(class_names))
print("Class names:", class_names)

# 3. Save the labels
class_path = "/content/class_names.json"
with open(class_path, "w") as f:
    json.dump(class_names, f, indent=2)

# 4. Package model and labels
zip_path = "/content/plant_disease_deployment.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(model_path, "best_mobilenetv2.keras")
    z.write(class_path, "class_names.json")

print("Deployment ZIP created:", zip_path)

# 5. Download the ZIP
files.download(zip_path)

# 6. Inspect preprocessing-related layers
print("\nModel layers:")
for layer in model.layers:
    print(layer.name, layer.__class__.__name__, layer.get_config().get("scale", ""))


## Conclusion and limitations
Three models (custom CNN, MobileNetV2, EfficientNetV2-B0) were trained on PlantDoc + PlantVillage images of eight diseases, using GPU-side augmentation, label smoothing, class weighting, two-stage fine-tuning and test-time augmentation. Results are reported separately on the original PlantDoc test set (realistic, headline) and a PlantVillage test set (optimistic).

**Limitations:** the PlantDoc test set has only 76 images (one image = ~1.3 points), so small differences between models are within noise. PlantVillage images come from controlled conditions and contain near-duplicates of the same leaf, so PlantVillage accuracy overstates real-world performance. The model only predicts among the eight selected classes; other plants, diseases or unusual backgrounds can be misclassified.